# Task 5: Decision Trees and Random Forests

**AI & ML Internship — Elevate Labs**

---

## Objective
Learn tree-based models for classification and regression using the Heart Disease dataset. Covers decision tree training, visualization, overfitting analysis, random forest comparison, feature importance interpretation, and cross-validation.

**Key goals:**
1. Load and explore the Heart Disease dataset
2. Train a Decision Tree Classifier and visualize the tree
3. Analyze overfitting and control tree depth
4. Train a Random Forest and compare accuracy
5. Interpret feature importances
6. Evaluate using cross-validation
7. Prepare for common tree-based model interview questions


## Dataset Description

**Dataset:** Heart Disease Dataset (Cleveland)  
**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Heart+Disease)  
**Local path:** `dataset/heart_disease.csv`  
**Shape:** 303 rows × 14 columns

### Columns
| Column | Type | Description |
|--------|------|-------------|
| age | int | Age in years |
| sex | int | Sex (1 = male, 0 = female) |
| cp | int | Chest pain type (0-3) |
| trestbps | int | Resting blood pressure |
| chol | int | Serum cholesterol |
| fbs | int | Fasting blood sugar > 120 mg/dl |
| restecg | int | Resting ECG results |
| thalach | int | Maximum heart rate achieved |
| exang | int | Exercise induced angina |
| oldpeak | float | ST depression induced by exercise |
| slope | int | Slope of peak exercise ST segment |
| ca | int | Number of major vessels colored by fluoroscopy |
| thal | int | Thalassemia type |
| target | int | **Target** — presence of heart disease (0 = no, 1 = yes) |

**Binary classification:** We will convert target to binary: 0 = no disease, 1 = disease.


## Import Libraries

- `pandas` — data loading and manipulation  
- `numpy` — numerical operations  
- `matplotlib.pyplot` — plotting  
- `seaborn` — statistical visualizations  
- `sklearn` — preprocessing, model training, evaluation, cross-validation
- `graphviz` — decision tree visualization


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully.")


## Load Dataset

Load the heart disease CSV with no header, then assign column names.

**Expected output:** Shape (303, 14)


In [ ]:

import os

dataset_path = "../dataset/heart_disease.csv"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(dataset_path, header=None, names=column_names)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")


## Initial Data Exploration

Inspect structure, data types, missing values, and class distribution.


In [ ]:

print("=== First 5 rows ===")
display(df.head())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values.")

print("\n=== Target Distribution ===")
print(df['target'].value_counts())
print(f"\nClass proportions:\n{df['target'].value_counts(normalize=True).round(3)}")

print("\n=== Summary Statistics ===")
display(df.describe().round(2))


## Data Preprocessing

**Steps:**
1. Convert target to binary (0 = no disease, 1 = disease)
2. Handle any missing values if present
3. Separate features and target
4. Split data into train and test sets
5. Standardize features

**Target encoding:** Original target has values 0, 1, 2, 3, 4. We convert to binary: 0 = no disease, 1 = disease.


In [ ]:

print("=== Preprocessing ===")

# Convert target to binary
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)
print(f"Target distribution after binarization:\n{df['target'].value_counts()}")

# Check for missing values
print(f"\nMissing values: {df.isnull().sum().sum()}")

# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

print(f"\nFeatures: {X.shape}, Target: {y.shape}")


## Train-Test Split and Standardization

Split first, then standardize only on training data.

**Split:** 80% train, 20% test  
**Random state:** 42 for reproducibility  
**Stratify:** Yes, by target to preserve class distribution


In [ ]:

print("=== Train-Test Split ===")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test class distribution: {pd.Series(y_test).value_counts().to_dict()}")

print("\n=== Standardization ===")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features standardized.")


## Decision Tree Classifier

**How decision trees work:**
1. Start with all data at root node
2. Find the best feature and threshold to split data into two groups
3. Repeat recursively for each child node
4. Stop when a stopping criterion is met (max depth, min samples, etc.)

**Key concepts:**
- **Entropy:** Measure of impurity/disorder in a node
- **Information Gain:** Reduction in entropy after a split
- **Gini Impurity:** Alternative to entropy; faster to compute
- **Max Depth:** Maximum number of levels in the tree


In [ ]:

print("=== Decision Tree Classifier ===")

# Train decision tree with default parameters
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)

# Predictions
y_pred_dt = dt.predict(X_test_scaled)
y_pred_proba_dt = dt.predict_proba(X_test_scaled)[:, 1]

# Evaluate
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_pred_proba_dt)

print(f"Decision Tree Performance:")
print(f"Accuracy: {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall: {recall_dt:.4f}")
print(f"F1-Score: {f1_dt:.4f}")
print(f"ROC-AUC: {roc_auc_dt:.4f}")
print(f"\nTree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}")


## Visualize Decision Tree

We visualize the trained decision tree to understand its decision-making process.

**Note:** The tree may be large. We limit the visualization depth for clarity.


In [ ]:

print("=== Decision Tree Visualization ===")

# Create a simpler tree for visualization
dt_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_simple.fit(X_train_scaled, y_train)

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_simple, feature_names=X_train.columns, class_names=['No Disease', 'Disease'], 
          filled=True, rounded=True, ax=ax, fontsize=10)
ax.set_title('Decision Tree Visualization (max_depth=3)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation: Each node shows the splitting condition, impurity, and class distribution. Leaves show the predicted class.")


## Overfitting Analysis

Decision trees tend to overfit when they grow too deep. We analyze this by comparing training and test accuracy at different tree depths.

**Overfitting symptoms:**
- Training accuracy keeps increasing
- Test accuracy plateaus or decreases
- Large gap between train and test performance


In [ ]:

print("=== Overfitting Analysis ===")

train_accuracies = []
test_accuracies = []
depths = range(1, 21)

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    train_accuracies.append(accuracy_score(y_train, dt.predict(X_train_scaled)))
    test_accuracies.append(accuracy_score(y_test, dt.predict(X_test_scaled)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(depths, train_accuracies, 'o-', color='steelblue', label='Training Accuracy', linewidth=2)
ax.plot(depths, test_accuracies, 'o-', color='coral', label='Test Accuracy', linewidth=2)
ax.axvline(x=5, color='gray', linestyle='--', label='Suggested max_depth=5')
ax.set_xlabel('Tree Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Overfitting Analysis: Train vs Test Accuracy', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

print("Interpretation: Training accuracy increases with depth, but test accuracy peaks then drops. The optimal depth is where test accuracy is highest.")


## Pruned Decision Tree

We apply hyperparameter tuning to prevent overfitting.

**Techniques:**
- Limit `max_depth`
- Require `min_samples_split` for a node to split
- Require `min_samples_leaf` for a leaf
- Use cost-complexity pruning (`ccp_alpha`)


In [ ]:

print("=== Pruned Decision Tree with Grid Search ===")

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dt_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Evaluate best model
best_dt = grid_search.best_estimator_
y_pred_best_dt = best_dt.predict(X_test_scaled)

accuracy_best_dt = accuracy_score(y_test, y_pred_best_dt)
precision_best_dt = precision_score(y_test, y_pred_best_dt)
recall_best_dt = recall_score(y_test, y_pred_best_dt)
f1_best_dt = f1_score(y_test, y_pred_best_dt)

print(f"\nPruned Decision Tree Performance:")
print(f"Accuracy: {accuracy_best_dt:.4f}")
print(f"Precision: {precision_best_dt:.4f}")
print(f"Recall: {recall_best_dt:.4f}")
print(f"F1-Score: {f1_best_dt:.4f}")


## Random Forest Classifier

Random Forest is an ensemble method that combines multiple decision trees.

**How it works:**
1. **Bagging:** Create multiple bootstrap samples from training data
2. **Random Feature Selection:** Each tree considers a random subset of features at each split
3. **Averaging:** Combine predictions from all trees (majority vote for classification)

**Why Random Forest is better than a single tree:**
- Reduces variance by averaging multiple trees
- Less prone to overfitting
- More robust to noise and outliers
- Provides feature importance out of the box


In [ ]:

print("=== Random Forest Classifier ===")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf = rf.predict(X_test_scaled)
y_pred_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

# Evaluate
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Random Forest Performance:")
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")
print(f"ROC-AUC: {roc_auc_rf:.4f}")


## Model Comparison

Compare Decision Tree vs Random Forest performance.


In [ ]:

print("=== Model Comparison ===")

comparison_data = {
    'Model': ['Decision Tree', 'Pruned DT', 'Random Forest'],
    'Accuracy': [accuracy_dt, accuracy_best_dt, accuracy_rf],
    'Precision': [precision_dt, precision_best_dt, precision_rf],
    'Recall': [recall_dt, recall_best_dt, recall_rf],
    'F1-Score': [f1_dt, f1_best_dt, f1_rf],
    'ROC-AUC': [roc_auc_dt, roc_auc_dt, roc_auc_rf]
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.round(4))

print("\nInterpretation: Random Forest typically outperforms single Decision Trees due to reduced variance and better generalization.")


## Feature Importance

Random Forest provides feature importance scores based on how much each feature decreases impurity across all trees.

**Interpretation:**
- Higher importance = feature is more useful for splitting
- Importance sums to 1 across all features
- Useful for feature selection and understanding model behavior


In [ ]:

print("=== Feature Importance (Random Forest) ===")

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

display(feature_importance)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis', ax=ax)
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation: Features at the top are most important for predicting heart disease. This can guide medical focus and feature engineering.")


## Cross-Validation

Cross-validation provides a more robust estimate of model performance by training on multiple train-validation splits.

**K-Fold Cross-Validation:**
1. Split data into K folds
2. Train on K-1 folds, validate on 1 fold
3. Repeat K times, each fold as validation once
4. Average performance across all K runs

**Benefits:**
- More reliable performance estimate
- Reduces variance in evaluation
- Uses data more efficiently


In [ ]:

print("=== Cross-Validation Comparison ===")

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

cv_results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'CV_Mean_Accuracy': scores.mean(),
        'CV_Std': scores.std(),
        'Min_Accuracy': scores.min(),
        'Max_Accuracy': scores.max()
    })
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_df = pd.DataFrame(cv_results)
display(cv_df.round(4))

print("\nInterpretation: Cross-validation gives a more robust estimate than single train-test split. Random Forest shows higher mean accuracy with lower variance.")


## Interview Preparation: Decision Trees & Random Forests Q&A

### 1. How does a decision tree work?

**Simple answer:** A decision tree splits data into smaller groups by asking yes/no questions, like a flowchart.

**Technical answer:**  
- **Algorithm:** Recursive binary splitting  
  1. Start with all samples at root node  
  2. For each feature and threshold, compute impurity reduction  
  3. Choose split that maximizes information gain or minimizes Gini impurity  
  4. Repeat recursively for child nodes  
  5. Stop when stopping criterion is met  
- **Stopping criteria:**  
  - `max_depth`: Maximum tree depth  
  - `min_samples_split`: Minimum samples to split a node  
  - `min_samples_leaf`: Minimum samples in a leaf  
  - `min_impurity_decrease`: Minimum improvement to split  
- **Leaf prediction:** Majority class (classification) or mean value (regression)

**Example:** To predict heart disease, the tree might first split on `age > 60`, then on `chol > 240`, creating a path of clinical decisions.

**Follow-up:** "What are entropy and Gini impurity, and how do they differ?"

---

### 2. What is entropy and information gain?

**Simple answer:** Entropy measures how mixed the classes are in a group. Information gain measures how much a split reduces that mixing.

**Technical answer:**  
- **Entropy:** `H(S) = -p_pos * log2(p_pos) - p_neg * log2(p_neg)`  
  - Ranges from 0 (pure) to 1 (maximally mixed)  
  - Measures disorder in a node  
- **Information Gain (IG):** `IG(S, A) = H(S) - H(S|A)`  
  - `H(S)`: Entropy before split  
  - `H(S|A)`: Weighted average entropy after split  
  - Higher IG = better split  
- **Gini Impurity:** `G(S) = 1 - sum(p_i^2)`  
  - Faster to compute than entropy  
  - Range: 0 (pure) to 0.5 (maximally mixed for binary)  
  - Often preferred in practice

**Example:** A node with 50 heart disease, 50 no disease has entropy 1.0. After splitting on `age > 60`, if left node has 5 disease/45 no-disease (entropy 0.54) and right has 45 disease/5 no-disease (entropy 0.54), IG = 1.0 - 0.54 = 0.46.

**Follow-up:** "When would you prefer Gini over entropy or vice versa?"

---

### 3. How is random forest better than a single tree?

**Simple answer:** Random Forest combines many trees to get a more reliable and stable prediction.

**Technical answer:**  
- **Bagging (Bootstrap Aggregating):**  
  - Create multiple bootstrap samples from training data  
  - Train a tree on each sample  
  - Average predictions (regression) or vote (classification)  
- **Random Feature Selection:**  
  - At each split, consider only a random subset of features  
  - Reduces correlation between trees  
  - Makes the ensemble more diverse  
- **Benefits over single tree:**  
  1. **Reduced variance:** Averaging reduces overfitting  
  2. **More stable:** Less sensitive to training data changes  
  3. **Better accuracy:** Often outperforms single trees  
  4. **Built-in feature importance:** Mean decrease in impurity  
  5. **Handles non-linearity:** Captures complex interactions  
- **Trade-offs:**  
  - Less interpretable than single tree  
  - Slower to train and predict  
  - More memory usage

**Example:** A single tree might predict heart disease based only on age. Random Forest averages many trees, each focusing on different features (age, cholesterol, ECG), giving a more robust prediction.

**Follow-up:** "What is the bias-variance tradeoff and how does Random Forest address it?"

---

### 4. What is overfitting and how do you prevent it?

**Simple answer:** Overfitting is when a model memorizes training data instead of learning patterns, performing well on training but poorly on new data.

**Technical answer:**  
- **Definition:** Model captures noise and specific patterns in training data that don't generalize  
- **Symptoms:**  
  - Training accuracy >> Test accuracy  
  - Large gap between train and test performance  
  - Model is too complex relative to data  
- **Prevention methods:**  
  1. **Tree pruning:** Limit `max_depth`, increase `min_samples_split`/`min_samples_leaf`  
  2. **Cross-validation:** Use CV to select optimal complexity  
  3. **Ensemble methods:** Random Forest, Gradient Boosting  
  4. **Regularization:** L1/L2 for linear models  
  5. **More data:** Reduces variance  
  6. **Feature selection:** Remove noisy/irrelevant features  
  7. **Early stopping:** Stop training when validation performance plateaus  
- **Detection:** Compare train vs test metrics; use learning curves

**Example:** A decision tree with `max_depth=None` might achieve 100% training accuracy but only 70% test accuracy. Pruning to `max_depth=5` might reduce training accuracy to 85% but increase test accuracy to 82%.

**Follow-up:** "How do you detect overfitting in practice?"

---

### 5. What is bagging?

**Simple answer:** Bagging creates multiple versions of a model by training on different subsets of data, then combines their predictions.

**Technical answer:**  
- **Bootstrap:** Sampling with replacement from training data  
  - Each bootstrap sample has same size as original  
  - ~63.2% of unique samples appear in each bootstrap sample  
  - ~36.8% are out-of-bag (OOB)  
- **Aggregating:** Combine predictions  
  - Classification: Majority vote  
  - Regression: Average  
- **Why it works:**  
  - Reduces variance without increasing bias  
  - Trees are high-variance, low-bias models  
  - Averaging high-variance models reduces variance  
- **Random Forest = Bagging + Random Feature Selection:**  
  - Standard bagging uses all features at each split  
  - Random Forest uses random subset, adding more diversity

**Example:** Training 100 decision trees on 100 different bootstrap samples. If 60 predict heart disease and 40 predict no disease, the ensemble predicts heart disease.

**Follow-up:** "What is the difference between bagging and boosting?"

---

### 6. How do you visualize a decision tree?

**Simple answer:** Use tree.plot_tree() or export to Graphviz to draw the decision flowchart.

**Technical answer:**  
- **Scikit-learn:** `sklearn.tree.plot_tree()`  
  - Built-in matplotlib-based visualization  
  - Shows nodes, splitting conditions, impurity, samples, and class distribution  
- **Graphviz:** `sklearn.tree.export_graphviz()`  
  - More customizable, higher quality  
  - Can export to PDF, PNG, DOT format  
  - Requires graphviz library  
- **What to visualize:**  
  - Node splitting condition (feature and threshold)  
  - Gini impurity or entropy at each node  
  - Number of samples in each node  
  - Class distribution in each node  
  - Predicted class at each leaf  
- **Best practices:**  
  - Limit depth for readability (max_depth=3-5)  
  - Use for small trees only  
  - For large trees, use feature importance instead

**Example:** A tree visualization might show: Root splits on `thalach <= 150`, left child splits on `age <= 55`, right child splits on `chol > 250`, etc.

**Follow-up:** "Why might you not want to visualize a very deep decision tree?"

---

### 7. How do you interpret feature importance?

**Simple answer:** Feature importance tells you which features the model relied on most for making predictions.

**Technical answer:**  
- **From Random Forest:** `model.feature_importances_`  
  - Based on mean decrease in impurity (Gini importance)  
  - Sums to 1 across all features  
  - Computed as total impurity decrease attributable to each feature  
- **From Decision Tree:** Same concept, but single tree  
- **Interpretation:**  
  - High importance = feature used often and effectively for splitting  
  - Low importance = feature rarely used or doesn't improve splits  
- **Limitations:**  
  - Biased toward high-cardinality features  
  - Correlated features share importance  
  - Doesn't indicate direction of effect  
- **Alternative methods:**  
  - Permutation importance: Shuffle feature and measure accuracy drop  
  - SHAP values: Game-theoretic feature contributions  
  - LIME: Local interpretable explanations

**Example:** If `thalach` has importance 0.25, it means 25% of all splits across all trees used `thalach`, and those splits were informative.

**Follow-up:** "What is permutation importance and how does it differ from Gini importance?"

---

### 8. What are the pros/cons of random forests?

**Simple answer:** Random Forests are accurate and robust but can be slow and hard to interpret.

**Technical answer:**  

**Pros:**
1. **High accuracy:** Often among best tabular models without tuning
2. **Reduced overfitting:** Less prone than single decision trees
3. **Robust to outliers:** Bootstrap sampling and averaging
4. **Handles non-linearity:** Captures complex interactions
5. **Feature importance:** Built-in interpretability tool
6. **Handles missing values:** Some implementations support it
7. **Parallelizable:** Trees are independent
8. **No need for scaling:** Tree-based, invariant to monotonic transformations

**Cons:**
1. **Less interpretable:** Black box compared to single tree
2. **Slower prediction:** Must query many trees
3. **More memory:** Stores hundreds of trees
4. **Biased feature importance:** Favors high-cardinality features
5. **Poor extrapolation:** Can't predict beyond training range
6. **Not online:** Can't update incrementally with new data
7. **May overfit noisy datasets:** Especially with many trees and deep trees

**Example:** Random Forest excels in medical diagnosis (heart disease) where accuracy and robustness matter more than interpretability. For credit scoring where regulations require explanations, a single decision tree may be preferred.

**Follow-up:** "When would you choose a Random Forest over Gradient Boosting or vice versa?"


## Conclusion

This notebook demonstrated tree-based models for classification:

1. **Loaded** the Heart Disease dataset (303 rows, 14 columns)
2. **Preprocessed** data: binarized target, standardized features
3. **Split** data into 80/20 train/test with stratification
4. **Trained** Decision Tree Classifier with default parameters
5. **Visualized** decision tree structure
6. **Analyzed** overfitting by varying tree depth
7. **Pruned** tree using GridSearchCV
8. **Trained** Random Forest Classifier
9. **Compared** Decision Tree vs Random Forest performance
10. **Interpreted** feature importances from Random Forest
11. **Evaluated** models using cross-validation
12. **Prepared** for 8 common tree-based model interview questions

### Key Takeaways
- Decision trees are intuitive but prone to overfitting
- Overfitting is controlled via max_depth, min_samples_split, min_samples_leaf
- Random Forest reduces variance through bagging and random feature selection
- Random Forest typically outperforms single decision trees
- Feature importance helps identify key predictors
- Cross-validation provides robust performance estimates
- Tree-based models don't require feature scaling
